# GRIDWISE AI - TRAFFIC COMMAND CENTER DEMO
## 14. Real-time Inference & Operational Recommendations

This notebook simulates the live production environment. As an incident arrives, we extract spatial features, run model inferences, compute the GridWise Operational Risk Index (GORI), and generate actionable dispatch recommendations.

In [1]:
import pandas as pd
import numpy as np
import time
import folium
import random
from IPython.display import display, HTML

### 1. The Inference Pipeline Engine

In [2]:
class CommandCenterAI:
    def __init__(self):
        # In a real deployment, load actual XGBoost/LightGBM models here via joblib
        print("Command Center Engine Initialized.")
        
    def predict_clearance_duration(self, features):
        # Simulate model inference (baseline 45 mins + penalties)
        base = 45
        if features['requires_road_closure']: base += 30
        if features['priority'] == 'High': base += 20
        # Add random noise for simulation
        duration = base + np.random.normal(0, 10)
        confidence = max(0.4, min(0.95, 1.0 - (duration / 200))) # Lower confidence for extremely long predictions
        return max(15, round(duration)), round(confidence * 100)
        
    def classify_deployment_load(self, features):
        # Simulate classification
        if features['requires_road_closure']: return "Heavy Response Unit", 88
        if features['priority'] == 'High': return "Standard Dispatch + Traffic Control", 75
        return "Standard Dispatch", 92
        
    def compute_gori_score(self, duration, is_hotspot):
        # GridWise Operational Risk Index (0-100)
        score = (duration / 120) * 50
        if is_hotspot: score += 30
        return min(100, round(score))
        
    def generate_action_recommendation(self, gori, duration):
        if gori > 80:
            return "CRITICAL: Escalate Manpower immediately. Major corridor diversion required."
        elif duration > 60:
            return "WARNING: Prolonged closure anticipated. Dispatch heavy tow units."
        else:
            return "Routine handling. Monitor for cascading congestion."
            
ai = CommandCenterAI()


Command Center Engine Initialized.


### 2. Simulating an Incoming Incident
A live ping hits the Command Center API from the traffic grid.

In [3]:
# Live Incident Payload
live_incident = {
    'id': 'EVT-2026-9901',
    'latitude': 12.971598,
    'longitude': 77.594562,
    'event_cause': 'Major Collision',
    'requires_road_closure': True,
    'priority': 'High',
    'hour_of_day': 17 # Rush Hour
}

print(f"🚨 INCOMING INCIDENT DETECTED: {live_incident['id']}")
print(f"Location: {live_incident['latitude']}, {live_incident['longitude']}")


🚨 INCOMING INCIDENT DETECTED: EVT-2026-9901
Location: 12.971598, 77.594562


### 3. Spatial Intelligence (Geo-Reconstruction)
Map coordinates to our historical DBSCAN clusters to flag hotspots.

In [4]:
def spatial_enrichment(lat, lon):
        # Simulate spatial lookup
        is_hotspot = True # Hardcoded for demo
        cluster_id = 4
        print(f"📍 GEO-INTELLIGENCE: Coordinates mapped to Historical Hotspot Cluster {cluster_id}")
        return is_hotspot, cluster_id

is_hotspot, geo_cluster = spatial_enrichment(live_incident['latitude'], live_incident['longitude'])


📍 GEO-INTELLIGENCE: Coordinates mapped to Historical Hotspot Cluster 4


### 4. Live AI Inference & Command Decisions

In [5]:
# 1. Predict Clearance Duration
predicted_mins, conf_reg = ai.predict_clearance_duration(live_incident)

# 2. Predict Deployment Load
deployment_class, conf_clf = ai.classify_deployment_load(live_incident)

# 3. Calculate Operational Risk (GORI)
gori_score = ai.compute_gori_score(predicted_mins, is_hotspot)

# 4. Generate AI Action Recommendation
action_rec = ai.generate_action_recommendation(gori_score, predicted_mins)

# Display Command Center Output
output_html = f"""
<div style="background-color: #1e1e1e; color: #00ffcc; padding: 20px; border-radius: 10px; font-family: monospace;">
    <h2 style="color: #ff3366; margin-top: 0;">⚡ GRIDWISE OPERATIONAL INTELLIGENCE ⚡</h2>
    <hr style="border-color: #333;">
    <p><b>EVENT ID:</b> {live_incident['id']} | <b>CAUSE:</b> {live_incident['event_cause']}</p>
    <br>
    <p>⏱️ <b>EST. CLEARANCE TIME:</b> <span style="color:white; font-size: 1.2em;">{predicted_mins} minutes</span> <i>(Confidence: {conf_reg}%)</i></p>
    <p>🚓 <b>DEPLOYMENT LOAD:</b> <span style="color:white; font-size: 1.2em;">{deployment_class}</span> <i>(Confidence: {conf_clf}%)</i></p>
    <p>🔥 <b>G.O.R.I SEVERITY INDEX:</b> <span style="color:{'#ff3366' if gori_score > 75 else '#ffcc00'}; font-size: 1.5em; font-weight: bold;">{gori_score} / 100</span></p>
    <br>
    <p style="background-color: #331111; padding: 10px; border-left: 5px solid #ff3366;">
        <b>🤖 AI ACTION RECOMMENDATION:</b><br>
        <span style="color: white;">{action_rec}</span>
    </p>
</div>
"""
display(HTML(output_html))


### 5. Tactical Mapping
Visualizing the incident relative to known traffic bottlenecks.

In [6]:
m = folium.Map(location=[live_incident['latitude'], live_incident['longitude']], zoom_start=15, tiles='CartoDB dark_matter')
folium.Marker(
    [live_incident['latitude'], live_incident['longitude']],
    popup=f"Event: {live_incident['id']}\nGORI: {gori_score}",
    icon=folium.Icon(color='red', icon='info-sign')
).add_to(m)
# Simulate hotspot radius
folium.Circle(
    radius=500,
    location=[live_incident['latitude'], live_incident['longitude']],
    color='crimson',
    fill=True,
).add_to(m)

display(m)
